In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, NMF
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr

SEED = 0
np.random.seed(SEED)

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()
gene_cols = pert_FC_selected.columns.tolist()
selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

In [4]:
# ---------------------------------------------------------------
# Prerequisites: Val-Split, Gene-Embeddings, Datensätze, Baseline
# ---------------------------------------------------------------

# 1. Val-Split von den Trainingsperturbationen abtrennen (vor allem Fitten!)
val_perts = list(np.random.choice(train_40, size=max(1, len(train_40) // 5), replace=False))
fit_perts = [p for p in train_40 if p not in val_perts]

# 2. Gene-Embeddings (nur auf fit_perts gefittet, damit nichts leaked)
def build_gene_embeddings(fc_df, train_perts, gene_cols, n_components=32):
    train_rows = fc_df.index.get_level_values("perturbation").isin(train_perts)
    train_block = fc_df.loc[train_rows]
    gene_signatures = train_block.T.values

    scaler = StandardScaler()
    gene_signatures = scaler.fit_transform(gene_signatures)

    n_components = min(n_components, gene_signatures.shape[0], gene_signatures.shape[1])
    pca = PCA(n_components=n_components, random_state=SEED)
    embeddings = pca.fit_transform(gene_signatures)

    return pd.DataFrame(embeddings, index=gene_cols), pca, scaler

def get_gene_embedding(gene, emb_df, dim):
    if gene in emb_df.index:
        return emb_df.loc[gene].values.astype(np.float32)
    return np.zeros(dim, dtype=np.float32)

gene_embeddings, gene_pca, gene_scaler = build_gene_embeddings(
    pert_FC_selected, fit_perts, gene_cols, n_components=32
)
EMB_DIM = gene_embeddings.shape[1]

# 3. Zielmatrix (nur fit_perts) für das NMF-Fitting
fit_mask = pert_FC_selected.index.get_level_values("perturbation").isin(fit_perts)
Y_fit_full = pert_FC_selected.loc[fit_mask].values

# 4. (perturbation, condition)-Datensätze bauen
def build_dataset(perturbations, fc_df, emb_df):
    Y_full, meta = [], []
    for pert in perturbations:
        sub = fc_df.loc[pert]
        for cond in sub.index.get_level_values("condition"):
            fc_vec = sub.loc[cond].values.astype(np.float32)
            Y_full.append(fc_vec)
            meta.append((pert, cond))
    return np.stack(Y_full).astype(np.float32), meta

_, meta_fit = build_dataset(fit_perts, pert_FC_selected, gene_embeddings)
Y_val_true, meta_val = build_dataset(val_perts, pert_FC_selected, gene_embeddings)
Y_test_true, meta_test = build_dataset(test_10, pert_FC_selected, gene_embeddings)

# 5. Baseline: Condition-Mittelwert der Trainingsperturbationen
def condition_mean_baseline(fc_df, train_perts):
    train_rows = fc_df.loc[fc_df.index.get_level_values("perturbation").isin(train_perts)]
    return train_rows.groupby(level="condition").mean()

baseline_means = condition_mean_baseline(pert_FC_selected, train_40)

In [5]:

# ---------------------------------------------------------------
# 1. NMF auf den Zielraum fitten (nur fit_perts, P/N-Split für Nicht-Negativität)
# ---------------------------------------------------------------
N_FACTORS = 10  # keep small due to, small training dataset, target dimension of gene vector

def to_nonneg(Y):
    return np.hstack([np.maximum(Y, 0), np.maximum(-Y, 0)]) # Turn data non-negative -> prerequisite for NMF

def from_nonneg(Y_nn, n_genes):
    return Y_nn[:, :n_genes] - Y_nn[:, n_genes:] # Backwards function

n_genes = len(gene_cols)
Y_fit_nn = to_nonneg(Y_fit_full) # make training data non-negative

nmf = NMF(n_components=N_FACTORS, init="nndsvda", random_state=SEED, max_iter=1000) # non-negative double single value decomposition variant "a" replaces zeros with the mean, is supposed to converge faster 
W_fit = nmf.fit_transform(Y_fit_nn)      # (n_fit_rows, N_FACTORS)
H = nmf.components_                       # (N_FACTORS, 2*n_genes)

# ---------------------------------------------------------------
# 2. KNN-Vorhersage im Faktorraum: neue Perturbation -> W über
#    Gene-Embedding-Ähnlichkeit zu den fit_perts schätzen
# ---------------------------------------------------------------
# one W-vektor per Pertubation (mean over conditions)
fit_pert_order = list(dict.fromkeys([p for p, c in meta_fit]))  # destinct order of pertubation names
W_per_pert = {}
for pert in fit_pert_order:
    idx = [i for i, (p, c) in enumerate(meta_fit) if p == pert] 
    W_per_pert[pert] = W_fit[idx].mean(axis=0) # dict of factor vectors for each pertubation. Uses the mean over the three conditions (Query gene-embedding is also not influenced by condition)

fit_gene_emb = np.stack([get_gene_embedding(p, gene_embeddings, EMB_DIM) for p in fit_pert_order]) # gene identity embeddings
fit_W = np.stack([W_per_pert[p] for p in fit_pert_order]) # matching nmf-factors

# Predict based on new, unseen pertubations
def knn_predict_W(query_perts, k=5):
    """Für jede Query-Perturbation: W = gewichteter Mittelwert der k ähnlichsten fit_perts."""
    query_emb = np.stack([get_gene_embedding(p, gene_embeddings, EMB_DIM) for p in query_perts]) # calculate gene embeddings
    nn = NearestNeighbors(n_neighbors=min(k, len(fit_pert_order)), metric="euclidean") # fit nearest neighbors on gene-embedding
    nn.fit(fit_gene_emb) # for every query pertubation, gives out the k nearest neighbours from fit_gene_emb:dist and their IDs
    dist, idx = nn.kneighbors(query_emb)

    weights = 1.0 / (dist + 1e-6) # weighing, closer neighbours get more weight than more distant ones, with division through zero prevented
    weights /= weights.sum(axis=1, keepdims=True)

    W_pred = np.stack([                                       # For each query pertubation get the factors of the k neighbors  
        (weights[i][:, None] * fit_W[idx[i]]).sum(axis=0)     # gives out a vector for each query 
        for i in range(len(query_perts))                      # this is the estimated factor activation for an unseen pertubation
    ])
    return W_pred

# ---------------------------------------------------------------
# 3. k über val_perts tunen (analog zum Early Stopping beim MLP)
# ---------------------------------------------------------------
def evaluate_knn(perts_list, X_meta, Y_true_full, k):
    unique_perts = list(dict.fromkeys([p for p, c in X_meta]))
    W_pred = knn_predict_W(unique_perts, k=k)
    W_pred_map = dict(zip(unique_perts, W_pred)) # extract exact pertubation-list for one split, and get their respective 
                                                 # predicted W. Save as dict

    preds = []
    for i, (pert, cond) in enumerate(X_meta):                   # per cell prediction of
        Y_nn_pred = W_pred_map[pert] @ H                        # matrix multiplication of the factor vector with the program matrix
        y_pred = from_nonneg(Y_nn_pred[None, :], n_genes)[0]    # transform backk to log2 space
        preds.append(y_pred)                                    # apped to list  
    preds = np.stack(preds)                                     
    mse = np.mean((Y_true_full - preds) ** 2)                   # calculate mse between predicted and actual expression values 
    return mse, preds

best_k, best_val_mse = None, np.inf                               # hyperparameter tuning
for k in [1, 3, 5, 8, 12]:
    val_mse, _ = evaluate_knn(val_perts, meta_val, Y_val_true, k)
    print(f"k={k:2d}  val_mse={val_mse:.4f}")
    if val_mse < best_val_mse:
        best_val_mse, best_k = val_mse, k

print(f"\nbest k = {best_k}")

# ---------------------------------------------------------------
# 4. Finale Evaluation auf test_10
# ---------------------------------------------------------------
_, test_preds = evaluate_knn(test_10, meta_test, Y_test_true, best_k)

rows = []
for i, (pert, cond) in enumerate(meta_test):
    y_true = Y_test_true[i]
    y_pred = test_preds[i]
    y_base = baseline_means.loc[cond].values

    r_model, _ = pearsonr(y_true, y_pred)
    r_base, _ = pearsonr(y_true, y_base)
    sp_model, _ = spearmanr(y_true, y_pred)
    sp_base, _ = spearmanr(y_true, y_base)

    rows.append(dict(
        perturbation=pert, condition=cond,
        pearson_model=r_model, pearson_baseline=r_base,
        mse_model=np.mean((y_true - y_pred) ** 2),
        mse_baseline=np.mean((y_true - y_base) ** 2),
        spearman_model=sp_model, spearman_baseline=sp_base,
    ))

results_nmf_knn = pd.DataFrame(rows)
print(results_nmf_knn.groupby("condition")[
    ["pearson_model", "pearson_baseline", "mse_model", "mse_baseline",
     "spearman_model", "spearman_baseline"]
].mean())

NMF reconstruction MSE (fit_perts): 0.0004
k= 1  val_mse=0.0022
k= 3  val_mse=0.0022
k= 5  val_mse=0.0021
k= 8  val_mse=0.0021
k=12  val_mse=0.0021

best k = 12
            pearson_model  pearson_baseline  mse_model  mse_baseline  \
condition                                                              
Co-culture       0.573117          0.592829   0.004299      0.003967   
Control          0.756701          0.754654   0.002482      0.002500   
IFNγ             0.715914          0.717705   0.002369      0.002430   

            spearman_model  spearman_baseline  
condition                                      
Co-culture        0.232671           0.267087  
Control           0.344404           0.398926  
IFNγ              0.304489           0.379795  


## Discussion
The model uses each perturbation's 20 surface-protein log2FC vector which is build exactly like the one for the RNA expression values and aims to predict the RNA log2FC vector compressed to 10 PCs using a PCA fit on the training perturbations. The learning algorithm we used was a ridge regression with an alpha chosen by LOOCV over the training perturbations. Via inverse-PCA the full log2FC vectors are retrieved from the predicted 10 PCs.

The model performance was measured using MSE as well as Pearson and Spearman correlation as we have used before for the baslione model. With a Pearson correlation of 0.690 vs 0.688, a spearman correlation of 0.348 vs 0.349, and an MSE of 0.00296 vs 0.00297 the model seems to perform just as well as the baseline model and does clearly not outperform it. Also the differences across the different condiotions behave similarly with Control performing best and Co-culture worst, which is also the most variable. This is also concordant with the structure of the model itself where the inverse transformation returns the mean log2FC vector + the corrections across the 10 PCs which via a high alpha (1000 in this case) are regularized strongly and thus shrink towards 0. This means the model is very similar to the mean-based baseline model. The fact there is a real CV minimum tells us that there is some real link between the protein data and the transcriptional response howerver this is very weak, possibly because the protein-panel is very small and most perturbations do not influence the expression of these surface proteins which mainly relate to immune-visibility and antigen-presentation. Also the 10 PCs capture only a part of the training variance (62-77%) under each condition so not all variance can be reconstructed from the predicted scores exactly 

Possibly more perturbations to train from could solve this problem as well as richer features which comprise a more diverse panel of proteins or also additional knowledge on the connection between genes and proteins could increase the model performance. However, overall it has been shown in the past that most complex models do not yet outperform very simple baseline models which predict the mean across perturbations (model 1) as shown by Constantin Ahlmann-Eltze, Wolfgang Huber & Simon Anders in 2025. 